# 5. Thời gian và tài nguyên crawler

## 5.1 Thiết kế phép đo và phạm vi

Số đo M1 được giữ từ một lượt Colab CPU-only đã lưu. Lượt đó có dải cấu hình 5.000 ID,
thực sự xử lý 14 paper trong khoảng đo rồi tạm dừng vì rate limit. Nó không phải lượt
crawling lại snapshot 73 paper, cũng không chứng minh đã hoàn tất toàn dải.

Runtime profiler đo các pha được theo dõi. Child wall đo tiến trình con đến lúc kết thúc;
phạm vi có thể gồm thêm bước ghi output sau khi profiler dừng. Thời gian cell validation
bên ngoài tiến trình không tự được tính vào child wall.

## 5.2 Môi trường và cấu hình thực tế

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
data = json.loads((Path.cwd().parent / 'report_summary.json').read_text(encoding='utf-8'))
scope, m1, m2 = data['scope'], data['m1'], data['m2']
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.spines.top': False, 'axes.spines.right': False})
print(f"Snapshot: {scope['papers']} paper / {scope['versions']} version; báo cáo ngày {data['report_date']}")

In [ ]:
run = data['crawler_measurement']
display(pd.Series(run['environment'], name='value').to_frame())
display(pd.DataFrame([{k:run[k] for k in ['status','runtime_sec','child_wall_sec','processed_papers','assigned_ids']}]))
display(pd.Series(run['intervals'], name='seconds').to_frame())
print('Các interval trên là cấu hình lịch sử, không phải mặc định hiện tại của source.')

## 5.3 Thời gian theo pha

Với tập paper thực sự xử lý trong run, trung bình profiler/paper là
$\overline{T}=T_{\mathrm{profiler}}/N_{\mathrm{processed}}$. Khi có child wall, có thể trình bày
thêm $T_{\mathrm{child}}/N_{\mathrm{processed}}$ và ghi rõ phạm vi. Không chia thời gian run cho
toàn bộ ID được giao hoặc toàn bộ record checkpoint cũ.

Các phase phản ánh instrumentation thực tế. Nếu có phase lồng nhau hoặc phần overhead
ngoài phase, tổng phase không nhất thiết bằng wall time. Không diễn giải chênh lệch đó
thành thời gian CPU hoặc I/O mà chưa có phép đo tương ứng.

In [ ]:
phases = pd.DataFrame(run['phases']).T; display(phases)
phases['elapsed_sec'].sort_values().plot.barh(xlabel='Giây', title='Thời gian pha trong lượt Colab đã lưu')
plt.tight_layout(); plt.show()
display(pd.DataFrame([{'profiler_sec_per_processed_paper':run['runtime_sec']/run['processed_papers'],
                      'child_wall_sec_per_processed_paper':run['child_wall_sec']/run['processed_papers'],
                      'wall_scope_difference_sec':run['child_wall_sec']-run['runtime_sec']}]))
display(Markdown(f"Pha references chiếm **{run['phases']['references']['elapsed_sec']/run['runtime_sec']:.1%}** "
                 'runtime profiler của lượt này. Đây là bằng chứng về chi phí references trong run đã đo, '
                 'không đủ để dự báo thời gian của toàn corpus.'))

## 5.4 RAM và dung lượng đĩa

Profiler lấy mẫu RSS của tiến trình chính và các tiến trình con tại cùng thời điểm.
RAM trung bình dùng trọng số theo khoảng thời gian giữa các mẫu, thay vì cộng peak riêng
của từng process. Đơn vị MB trong stats tương ứng $1024^2$ byte.

Peak disk là cực đại trong các lần lấy mẫu và checkpoint thao tác trên thư mục được theo
dõi, không phải một cận trên tuyệt đối của mọi byte tồn tại. Final disk gồm working data
và checkpoint thuộc phạm vi đo. Backup Drive hoặc log ngoài phạm vi theo dõi phải được
tính riêng nếu muốn trình bày dung lượng toàn hệ thống.

In [ ]:
display(pd.Series(run['memory'], name='RAM').to_frame())
disk = dict(run['disk'])
for field in ['peak_bytes','final_bytes']:
    disk[field.replace('_bytes','_MiB')] = disk[field]/1024**2
display(pd.Series(disk, name='disk').to_frame())

## 5.5 Diễn giải và điều chỉnh

Request limit, phân trang, kích thước archive, cache và retry khiến chi phí giữa các paper
khác nhau. Tăng batch metadata/S2 giảm số lượt gọi ở những chỗ phù hợp; nó không loại bỏ
thời gian tải từng version và không bảo đảm tốc độ tăng cùng tỷ lệ.

Lượt lưu có interval source 1 giây. Source mới mặc định 3 giây theo cấu hình đã điều chỉnh;
số đo cũ vẫn giữ nguyên 1 giây. Muốn so sánh hai cấu hình cần chạy các lượt thật có scope,
input và cache được công bố. Không sửa log để biến một cấu hình thành cấu hình khác.

## Tài liệu và mã tham chiếu

- [Đặc tả M1](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/).
- [arXiv API: cấu trúc truy vấn và Atom](https://info.arxiv.org/help/api/user-manual.html).
- [Quy định truy cập arXiv](https://info.arxiv.org/help/api/tou.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã triển khai: [src/crawler/profiler.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/e8d409200fa3483bc52eb5a914c602c552def133/src/crawler/profiler.py).
- Mã triển khai: [src/crawler/main.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/e8d409200fa3483bc52eb5a914c602c552def133/src/crawler/main.py).
- Mã triển khai: [src/crawler/request_policy.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/e8d409200fa3483bc52eb5a914c602c552def133/src/crawler/request_policy.py).
- Mã triển khai: [colab_crawler.ipynb](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/e8d409200fa3483bc52eb5a914c602c552def133/colab_crawler.ipynb).